In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Architecture

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, heads, attn_dropout=0.0,resid_drop=0.0,qkv_bias=True):
    super(MultiHeadAttention,self).__init__()
    assert(d_model % heads == 0)
    self.d_model = d_model
    self.heads = heads
    self.head_dim = d_model // heads

    self.query=nn.Linear(d_model,d_model,bias=qkv_bias)
    self.key=nn.Linear(d_model,d_model,bias=qkv_bias)
    self.value=nn.Linear(d_model,d_model,bias=qkv_bias)
    self.out_fc=nn.Linear(d_model,d_model,bias=qkv_bias)
    self.attn_dropout=nn.Dropout(attn_dropout)
    self.resid_dropout=nn.Dropout(resid_drop)



  def forward(self,q,k,v,mask=None):
    batch_size=q.shape[0]
    q_len,k_len,v_len=q.shape[1],k.shape[1],v.shape[1]

    q=self.query(q).view(batch_size,q_len,self.heads,self.head_dim)
    k=self.key(k).view(batch_size,k_len,self.heads,self.head_dim)
    v=self.value(v).view(batch_size,v_len,self.heads,self.head_dim)

    energy=torch.einsum("nqhd,nkhd->nhqk",[q,k])

    if mask is not None:
      energy=energy.masked_fill(mask==0,-torch.inf)

    attention=torch.softmax(energy/(self.head_dim**(1/2)),dim=-1)
    attention=self.attn_dropout(attention)

    out=torch.einsum("nhql,nlhd->nqhd",[attention,v])
    out=out.reshape(batch_size,q_len,self.d_model)
    out=self.out_fc(out)
    out=self.resid_dropout(out)
    return out

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,d_model,dropout=0.0,bias=True):
    super(FeedForward,self).__init__()
    self.layers=nn.Sequential(
        nn.Linear(d_model,d_model*4,bias=bias),
        nn.GELU(),
        nn.Linear(d_model*4,d_model,bias=bias)
    )
    self.dropout=nn.Dropout(dropout)

  def forward(self,x):
    x=self.layers(x)
    x=self.dropout(x)
    return x

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super(TransformerBlock,self).__init__()
    self.norm1=nn.LayerNorm(cfg.d_model,bias=cfg.bias)
    self.attn=MultiHeadAttention(cfg.d_model,cfg.heads,cfg.attn_drop,cfg.resid_drop,cfg.bias)
    self.norm2=nn.LayerNorm(cfg.d_model,bias=cfg.bias)
    self.ff=FeedForward(cfg.d_model,cfg.resid_drop,bias=cfg.bias)

  def forward(self,x,mask=None):
    shortcut=x
    x=self.norm1(x)
    x=shortcut+self.attn(x,x,x,mask)
    x=x+self.ff(self.norm2(x))
    return x

In [ ]:
class GPT2(nn.Module):
  def __init__(self,cfg):
    super(GPT2,self).__init__()
    assert cfg.max_len is not None
    assert cfg.vocab_size is not None
    self.cfg=cfg

    self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
    self.pos_emb=nn.Embedding(cfg.max_len,cfg.d_model)
    self.drop=nn.Dropout(cfg.embed_drop)

    self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])

    self.norm=nn.LayerNorm(cfg.d_model,bias=cfg.bias)
    self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=cfg.bias)

    self.pad_idx=cfg.pad_id

    if cfg.weight_tying:
      self.tok_emb.weight=self.head.weight

  def make_causal_mask(self,x):
    device=x.device
    batch_size, seq_len = x.shape
    pad_mask = (x != self.pad_idx).unsqueeze(1).unsqueeze(2)
    causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)
    return (pad_mask & causal_mask).to(device)

  def forward(self,x):
    device=x.device
    batch_size,seq_len=x.shape
    pos = torch.arange(0, seq_len, dtype=torch.long, device=device)
    mask=self.make_causal_mask(x)

    tok_emb=self.tok_emb(x)
    pos_emb=self.pos_emb(pos)
    x=self.drop(tok_emb+pos_emb)

    for block in self.blocks:
      x = block(x, mask)

    x=self.norm(x)
    logits=self.head(x)
    return logits

# Definitions and Tokenizer

In [ ]:
def calc_loss_batch(input_batch,target_batch,model,device,label_smoothing=0.0):
  input_batch=input_batch.to(device)
  target_batch=target_batch.to(device)
  logits=model(input_batch)

  loss=F.cross_entropy(
      logits.flatten(0,1),target_batch.flatten(), label_smoothing=label_smoothing
  )

  return loss

In [ ]:
def calc_loss_loader(data_loader,model,device,num_batches=None):
  total_loss=0
  if len(data_loader) == 0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))

  for i,(src,trg) in enumerate(data_loader):
    if i >= num_batches:
      break

    loss=calc_loss_batch(src,trg,model,device)
    total_loss+=loss.item()

  return total_loss/num_batches

In [ ]:
@torch.no_grad()
def generate(self, idx, max_new_tokens, max_len, temperature=1.0, top_k=None,device='cpu'):
  self.to(device)
  idx=idx.to(device)

  for _ in range(max_new_tokens):
      idx_cond = idx if idx.size(1) <= max_len else idx[:, -max_len:]
      logits = self(idx_cond)
      logits = logits[:, -1, :]
      if top_k is not None:
          v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
          logits[logits < v[:, [-1]]] = -float('Inf')

      if temperature > 0.0:
          logits = logits / temperature
          probs = F.softmax(logits, dim=-1)
          idx_next = torch.multinomial(probs, num_samples=1)
      else:
          idx_next = torch.argmax(logits, dim=-1, keepdim=True)

      if idx_next.item() == EOS_ID:
          break

      idx = torch.cat((idx, idx_next), dim=1)

  return idx

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import tiktoken

tokenizer=tiktoken.get_encoding("gpt2")

BOS_ID = tokenizer.n_vocab
PAD_ID = tokenizer.n_vocab + 1
EOS_ID = tokenizer.eot_token

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [ ]:
class GPTConfig:
  max_len: int = 1024
  vocab_size: int = tokenizer.n_vocab # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
  layers: int = 12
  heads: int = 12
  d_model: int = 768
  attn_drop: float = 0.1
  resid_drop: float = 0.1
  embed_drop: float = 0.1
  bias: bool = True
  weight_tying: bool = False
  pad_id: int = PAD_ID
  eos_id: int = EOS_ID

# Download

In [ ]:
import urllib.request
import json
import tensorflow as tf

def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    backup_base_url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        backup_url = os.path.join(backup_base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)

        if not os.path.exists(file_path):
          download_file(file_url, file_path, backup_url)

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params

def download_file(url, destination, backup_url=None):
  def _attempt_download(download_url):
      with urllib.request.urlopen(download_url) as response:
          # Get the total file size from headers, defaulting to 0 if not present
          file_size = int(response.headers.get("Content-Length", 0))

          # Check if file exists and has the same size
          if os.path.exists(destination):
              file_size_local = os.path.getsize(destination)
              if file_size == file_size_local:
                  print(f"File already exists and is up-to-date: {destination}")
                  return True  # Indicate success without re-downloading

          block_size = 1024  # 1 Kilobyte

          # Initialize the progress bar with total file size
          progress_bar_description = os.path.basename(download_url)
          with tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
              with open(destination, "wb") as file:
                  while True:
                      chunk = response.read(block_size)
                      if not chunk:
                          break
                      file.write(chunk)
                      progress_bar.update(len(chunk))
          return True

  try:
      if _attempt_download(url):
          return
  except (urllib.error.HTTPError, urllib.error.URLError):
      if backup_url is not None:
          print(f"Primary URL ({url}) failed. Attempting backup URL: {backup_url}")
          try:
              if _attempt_download(backup_url):
                  return
          except urllib.error.HTTPError:
              pass

      # If we reach here, both attempts have failed
      error_message = (
          f"Failed to download from both primary URL ({url})"
          f"{' and backup URL (' + backup_url + ')' if backup_url else ''}."
          "\nCheck your internet connection or the file availability.\n"
          "For help, visit: https://github.com/rasbt/LLMs-from-scratch/discussions/273"
      )
      print(error_message)
  except Exception as e:
      print(f"An unexpected error occurred: {e}")

def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params

In [ ]:
settings, params = download_and_load_gpt2("124M", "/content/drive/MyDrive/gpt2")

print("Settings:", settings)
print("Params:", params)

In [ ]:
# %%script echo skip

def assign(left, right):
  if left.shape != right.shape:
    raise ValueError(f"Shapes do not match: {left.shape} vs {right.shape}")

  return nn.Parameter(torch.tensor(right, dtype=torch.float32))

In [ ]:
def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.blocks[b].attn.query.weight = assign(
            gpt.blocks[b].attn.query.weight, q_w.T)
        gpt.blocks[b].attn.key.weight = assign(
            gpt.blocks[b].attn.key.weight, k_w.T)
        gpt.blocks[b].attn.value.weight = assign(
            gpt.blocks[b].attn.value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.blocks[b].attn.query.bias = assign(
            gpt.blocks[b].attn.query.bias, q_b)
        gpt.blocks[b].attn.key.bias = assign(
            gpt.blocks[b].attn.key.bias, k_b)
        gpt.blocks[b].attn.value.bias = assign(
            gpt.blocks[b].attn.value.bias, v_b)

        gpt.blocks[b].attn.out_fc.weight = assign(
            gpt.blocks[b].attn.out_fc.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.blocks[b].attn.out_fc.bias = assign(
            gpt.blocks[b].attn.out_fc.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.blocks[b].ff.layers[0].weight = assign(
            gpt.blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.blocks[b].ff.layers[0].bias = assign(
            gpt.blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.blocks[b].ff.layers[2].weight = assign(
            gpt.blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.blocks[b].ff.layers[2].bias = assign(
            gpt.blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.blocks[b].norm1.weight = assign(
            gpt.blocks[b].norm1.weight,
            params["blocks"][b]["ln_1"]["g"])
        gpt.blocks[b].norm1.bias = assign(
            gpt.blocks[b].norm1.bias,
            params["blocks"][b]["ln_1"]["b"])
        gpt.blocks[b].norm2.weight = assign(
            gpt.blocks[b].norm2.weight,
            params["blocks"][b]["ln_2"]["g"])
        gpt.blocks[b].norm2.bias = assign(
            gpt.blocks[b].norm2.bias,
            params["blocks"][b]["ln_2"]["b"])

    gpt.norm.weight = assign(gpt.norm.weight, params["g"])
    gpt.norm.bias = assign(gpt.norm.bias, params["b"])
    gpt.head.weight = assign(gpt.head.weight, params["wte"])


    if gpt.cfg.weight_tying:
        gpt.tok_emb.weight = gpt.head.weight

# Test

In [ ]:
model=GPT2(GPTConfig())
model.eval()

In [ ]:
# %%script echo skip

load_weights_into_gpt(model, params)
model.to(device)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

text_area = widgets.Textarea(
    placeholder='영어 문장을 입력하세요',
    layout=widgets.Layout(width='600px', height='100px')
)
button = widgets.Button(description="Enter")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        prompt = text_area.value
        if not prompt.strip():
            print("문장을 입력해주세요.")
            return

        token_ids = generate(
            model,
            text_to_token_ids(prompt, tokenizer),
            max_new_tokens=128,
            max_len=128,
            temperature=1.0,
            top_k=10,
            device=device
        )

        print("입력 텍스트: ", prompt)
        print("-"*80)
        print("출력 텍스트: ", token_ids_to_text(token_ids, tokenizer))

button.on_click(on_click)
display(text_area, button, output)

# 결과

입력 텍스트:  The Transformer is a neural network architecture introduced in 2017. Unlike recurrent neural networks, it does not process a sequence strictly from left to right. Instead, it relies on a mechanism known as self-attention, which allows each token to interact with other tokens in the sequence. This architecture was originally proposed for
--------------------------------------------------------------------------------
출력 텍스트:  *The Transformer is a neural network architecture introduced in 2017. Unlike recurrent neural networks, it does not process a sequence strictly from left to right. Instead, it relies on a mechanism known as self-attention, which allows each token to interact with other tokens in the sequence. This architecture was originally proposed for* a network in the 1980s by researchers from Princeton University.

The Transformer's first step was to build a neural network with all the information of a regular, distributed, and self-contained token, and use a set of rules for its execution. The first rule was to create a neural network that could perform all the processing necessary to generate a set of tokens, and a new rule for its evaluation would be created.

The second rule is for the transformer's output to be distributed to a set of nodes, which would then process the output and decide which tokens would be valid, or not valid. The rule itself

# --------------TO DO LIST--------------------

PAD_ID EOS_ID 적기

model 초기화

외부 파라미터 로드

마무리 짓기